# Studying SSB as an Operator Replacement

This notebook answers a narrow question: can the runtime operator layer be used to implement a spontaneous-symmetry-breaking field shift such as

$$\phi \to v + h?$$

Short answer:

- **`FieldOperator` / `replacement_operator(...)`** are **not** finite substitutions. They compute a slot-wise operator action with a Leibniz sum, so they are the right tool for infinitesimal variations, not for a full broken-phase rewrite.
- A **custom `TermOperator`** can implement a finite shift, but that logic has to be written explicitly at the whole-term level.
- For the electroweak broken phase, the repository's dedicated SSB helpers remain the intended public path.


In [ ]:
from dataclasses import replace
from itertools import product
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
for p in (ROOT, SRC):
    s = str(p)
    if s not in sys.path:
        sys.path.insert(0, s)

from symbolica import Expression, S

from model import Model, PartialD, scalar_field
from model.interactions import DerivativeAction
from lagrangian.operator_action import (
    FieldOperator,
    OperatorAtomResult,
    OperatorSummand,
    TermOperator,
)


def show(label, lagrangian):
    print(label)
    print(lagrangian.to_symbolica())
    print()


## 1. A toy scalar model

We use one real scalar with a standard kinetic term and a quartic potential,

$$\mathcal{L} = \frac12 (\partial_\mu \phi)(\partial^\mu \phi) - \frac{m^2}{2}\phi^2 + \frac{\lambda}{4}\phi^4.$$

The shift we want to study is

$$\phi \to v + h,$$

where `v` is a constant vev and `h` is the fluctuation field.


In [ ]:
mu = S("mu")
v = S("v")
m2 = S("m2")
lam = S("lam")
half = Expression.num(1) / 2
quarter = Expression.num(1) / 4

phi = scalar_field("phi", self_conjugate=True)
h = scalar_field("h", self_conjugate=True)

model_potential = Model((-m2 * half) * phi * phi + (lam * quarter) * phi * phi * phi * phi)
L_potential = model_potential.lagrangian()

model_full = Model(
    half * PartialD(phi, mu) * PartialD(phi, mu)
    + (-m2 * half) * phi * phi
    + (lam * quarter) * phi * phi * phi * phi
)
L_full = model_full.lagrangian()

show("Potential only:", L_potential)
show("Full toy model:", L_full)


## 2. Why a `FieldOperator` is not the SSB shift

The natural first attempt is to tell the slot-wise operator layer that one `phi` can become either `h` or the constant `v`.

That is **not** the finite substitution $\phi \to v + h$. It is the operator action

$$O[\phi] = h + v,$$

followed by the graded Leibniz sum over term positions. So on `phi^4` it replaces **one slot at a time**, not all four slots independently.


In [ ]:
linearized_shift = FieldOperator(
    "linearized_shift",
    on_field=lambda occ: (
        OperatorAtomResult(
            summands=(
                OperatorSummand(replacement=(h(),)),
                OperatorSummand(coefficient=v, replacement=()),
            )
        )
        if occ.field is phi
        else None
    ),
)

show("FieldOperator acting on the potential:", L_potential.apply_operator(linearized_shift))

try:
    show("FieldOperator acting on the full model:", L_full.apply_operator(linearized_shift))
except Exception as exc:
    print("FieldOperator fails on the kinetic term:")
    print(type(exc).__name__, exc)


The two important observations are:

1. On the potential, the result is only the **linearized one-slot response**. It is not the full polynomial expansion of $(v+h)^2$ or $(v+h)^4$.
2. On the kinetic term, the constant branch is rejected because the current slot-wise engine does not allow a differentiated slot to be replaced by the empty tuple. Physically $\partial_\mu v = 0$, but that rule is not encoded in the generic `FieldOperator` path.


## 3. A notebook-local `TermOperator` for a finite scalar shift

For a simple real scalar, we can write the finite substitution explicitly as a **whole-term** rewrite.

The helper below is intentionally narrow:

- one self-conjugate scalar species `phi`
- replacement `phi -> v + h`
- no fermions, ghosts, or closed Dirac bilinears
- if a differentiated `phi` takes the constant branch, that branch is dropped so that `partial(v) = 0`

This is enough to study the mechanism, but it is not a general public SSB API.


In [ ]:
def finite_scalar_shift_term_operator(name, *, source_field, shifted_field, vev):
    def apply_to_term(term):
        if term.closed_dirac_bilinears:
            raise ValueError("This notebook helper only supports scalar terms with no Dirac bilinears.")
        for occ in term.fields:
            if occ.field.kind != "scalar":
                raise ValueError("This notebook helper only supports scalar fields.")

        derivatives_by_slot = {}
        for action in term.derivatives:
            derivatives_by_slot.setdefault(action.target, []).append(action)

        target_slots = []
        choices = []
        for slot, occ in enumerate(term.fields):
            if occ.field is not source_field:
                continue
            target_slots.append(slot)
            if derivatives_by_slot.get(slot):
                choices.append(((shifted_field.occurrence(labels=occ.labels), Expression.num(1), True),))
            else:
                choices.append(
                    (
                        (shifted_field.occurrence(labels=occ.labels), Expression.num(1), True),
                        (None, vev, False),
                    )
                )

        if not choices:
            return (term,)

        outputs = []
        for assignment in product(*choices):
            assignment_by_slot = dict(zip(target_slots, assignment))
            coefficient = term.coupling
            new_fields = []
            old_to_new = {}

            for old_slot, occ in enumerate(term.fields):
                branch = assignment_by_slot.get(old_slot)
                if branch is None:
                    old_to_new[old_slot] = len(new_fields)
                    new_fields.append(occ)
                    continue

                replacement_occ, factor, keep_slot = branch
                coefficient = coefficient * factor
                if keep_slot:
                    old_to_new[old_slot] = len(new_fields)
                    new_fields.append(replacement_occ)

            new_derivatives = []
            valid = True
            for action in term.derivatives:
                if action.target not in old_to_new:
                    valid = False
                    break
                new_derivatives.append(
                    DerivativeAction(
                        target=old_to_new[action.target],
                        lorentz_index=action.lorentz_index,
                    )
                )
            if not valid:
                continue

            outputs.append(
                replace(
                    term,
                    coupling=coefficient,
                    fields=tuple(new_fields),
                    derivatives=tuple(new_derivatives),
                )
            )
        return tuple(outputs)

    return TermOperator(name=name, apply_to_term=apply_to_term)


finite_shift = finite_scalar_shift_term_operator(
    "finite_shift_phi_to_v_plus_h",
    source_field=phi,
    shifted_field=h,
    vev=v,
)


In [ ]:
show("Finite TermOperator shift on the potential:", L_potential.apply_operator(finite_shift))
show("Finite TermOperator shift on the full model:", L_full.apply_operator(finite_shift))


## 4. Interpretation

For this toy scalar example, the `TermOperator` route does what we want:

- the potential is expanded as a genuine finite substitution in `v` and `h`
- the kinetic term becomes $\frac12 (\partial h)^2$ because branches with `partial(v)` are discarded

But this works only because we wrote the whole-term logic ourselves.


## 5. Conclusion

- **Can a runtime replacement achieve SSB directly?**
  Not with the generic slot-wise `FieldOperator` path. That path computes a derivation-like action, not a finite field substitution.
- **Can it be done at the compiled-term layer anyway?**
  Yes, but only with a custom whole-term rewrite such as the notebook-local `TermOperator` above.
- **Is that the recommended route for the electroweak model?**
  No. The repository already has dedicated broken-phase helpers in `src/model/ssb.py` and the walkthrough in `notebooks/electroweak_ssb_walkthrough.ipynb`.

So the clean statement is:

**operator replacement is good for variations; a full SSB shift is a finite rewrite and belongs either in a custom `TermOperator` or in the dedicated SSB layer.**
